# Hard-coding a single iteration with sparse operations
We will develop cupy code for a single iteration of our sparse tensor factorization model.
We build upon the previous notebook where we developed a first dot product with sparse operations (which is the main bottleneck of the algorithm).

In [1]:
# we reload the packages as we make changes frequently
%load_ext autoreload
%autoreload 2

In [2]:
from tensorly_custom.decomposition import non_negative_tucker
import tensorly_custom as tl
import numpy as np
import pickle
import torch
import os
from typing import List, Tuple
from utils import DATA_DIR, select_gpu, tree_to_device
from typing import Optional, Union
import cupy as cp
import cupyx.scipy.sparse as cpx_sparse
import sparse
import tensorflow as tf

2025-11-27 09:47:30.269344: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
device = select_gpu(2)
def _to_np(x):
    # Accept NumPy arrays or torch tensors; return NumPy view/copy
    if hasattr(x, "detach"):  # torch.Tensor
        return x.detach().cpu().numpy()
    return x

def _torch_or_tucker_load(path, map_location="cpu"):
    """Tries to load a torch-saved file, if fails, tries pickle."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except RuntimeError:
        with open(path, "rb") as f:
            return pickle.load(f)

def cupy_to_torch_sparse(
    cu_mat: cpx_sparse.spmatrix,
    orig_shape: Optional[Tuple[int, ...]] = None,
    device: Union[str, torch.device] = "cpu",
    dtype: Optional[torch.dtype] = None,
) -> torch.Tensor:
    """
    Convert a CuPy sparse matrix (any format) back to a torch sparse COO tensor.

    If orig_shape is None:
        - The torch tensor is 2D and has the same shape as cu_mat.
    If orig_shape is provided and len(orig_shape) == 2:
        - The torch tensor is 2D with that shape.
    If orig_shape is provided and len(orig_shape) > 2:
        - We treat cu_mat.row as the flattened N-D index and unflatten it
          back to N-D using np.unravel_index, assuming the representation
          created by `torch_sparse_to_cupy`.

    Args:
        cu_mat: CuPy sparse matrix (COO/CSR/CSC, will be converted to COO).
        orig_shape: original tensor shape (for N-D tensors).
        device: target torch device.
        dtype: target dtype for values (defaults to inferred from data).

    Returns:
        torch.sparse_coo_tensor on the requested device.
    """
    # Ensure COO format
    if not cpx_sparse.isspmatrix_coo(cu_mat):
        cu_mat = cu_mat.tocoo()

    row_cp = cu_mat.row
    col_cp = cu_mat.col
    data_cp = cu_mat.data

    # Bring back to NumPy on host
    row_np = cp.asnumpy(row_cp)
    col_np = cp.asnumpy(col_cp)
    data_np = cp.asnumpy(data_cp)

    if orig_shape is None:
        # Simple 2D case, use cu_mat.shape directly
        shape = cu_mat.shape
        indices_np = np.vstack([row_np, col_np])
    else:
        shape = tuple(orig_shape)
        if len(shape) == 2:
            # 2D round-trip
            indices_np = np.vstack([row_np, col_np])
        else:
            # N-D round-trip: row_np contains flattened indices
            flat = row_np
            coords = np.unravel_index(flat, shape)  # tuple of arrays
            indices_np = np.vstack(coords)          # shape: (ndim, nnz)

    indices_t = torch.from_numpy(indices_np).long()
    values_t = torch.from_numpy(data_np)

    if dtype is not None:
        values_t = values_t.to(dtype)

    # Build sparse tensor
    x = torch.sparse_coo_tensor(indices_t, values_t, size=shape)
    x = x.coalesce()
    x = x.to(device)

    return x
def torch_sparse_to_cupy(
    x: torch.Tensor,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Convert a torch sparse COO tensor to a CuPy COO sparse matrix.

    For 2D tensors, the mapping is straightforward.
    For N-D tensors (N>2), we flatten the N-D indices to a single row index
    using np.ravel_multi_index and store them in a (prod(shape), 1) matrix.

    The original shape is returned for reconstruction.
    Returns:
        (cupy_coo_matrix, original_shape)
    """
    if not x.is_sparse:
        raise TypeError("torch_sparse_to_cupy expects a torch sparse tensor (COO).")

    x = x.coalesce()  # ensure indices are unique & sorted
    indices = x.indices()          # shape: (ndim, nnz)
    values = x.values()            # shape: (nnz,)
    shape = tuple(x.shape)

    # Move to CPU and NumPy
    indices_np = indices.cpu().numpy()
    values_np = values.cpu().numpy()

    ndim, nnz = indices_np.shape

    if ndim == 2:
        # Direct 2D mapping
        row = indices_np[0]
        col = indices_np[1]
        row_cp = cp.asarray(row)
        col_cp = cp.asarray(col)
        data_cp = cp.asarray(values_np)
        cu_mat = cpx_sparse.coo_matrix((data_cp, (row_cp, col_cp)), shape=shape)
    else:
        # Flatten N-D indices to a single dimension (row index)
        coords = [indices_np[d] for d in range(ndim)]
        flat = np.ravel_multi_index(coords, shape)  # shape: (nnz,)
        flat_cp = cp.asarray(flat)
        data_cp = cp.asarray(values_np)
        # Store as a (prod(shape), 1) matrix: column index always 0
        zero_cp = cp.zeros_like(flat_cp)
        cu_mat = cpx_sparse.coo_matrix(
            (data_cp, (flat_cp, zero_cp)),
            shape=(int(np.prod(shape)), 1),
        )

    return cu_mat, shape


def _role_index(role: str) -> int:
    if role == "verb":
        return 0
    elif role == "subject":
        return 1
    elif role == "object":
        return 2
    else:
        raise ValueError("role must be one of {'verb','subject','object'}")


class SparseTupleTensor:
    """Encapsulating the Sparse TupleTensor (built from vectors extracted from corpus) and the vocabulary,
    providing methods for decomposition, refactoring, etc.."""
    def __init__(self, tensor, device="cpu", sparsity_type=None):
        self.tensor = tensor
        self.sparsity_type = sparsity_type
        self.shape = tensor.shape
        self.device = device

    # --- Construction and loading ---
    @classmethod
    def load_from_disk(cls,
                       dataset: str="karrewiet_sparse",
                       method: str="counting",
                       dims: int=1000,
                       map_location: str="cpu",
                       sparsity_type: Optional[str]=None
                          ) -> "SparseTupleTensor":

        """Loads a precomputed tucker decomposition from disk.
            Args:
                dataset (str): name of the dataset
                method (str): method used to compute the decomposition
                    - one of "counting", "sc", "sii"
                dims (int): dimensionality of the original tensor modes (vocab size)
                rank (int): rank of the decomposition
                iterations (int): number of iterations used to compute the decomposition
                map_location (str): device to map the loaded tensors to
            Returns:
                ((core, factors), vocab)
                    core: torch.Tensor
                    factors: list[torch.Tensor]
                    vocab: dict with keys 'vocab_v','vocab_s','vocab_o','v2i','s2i','o2i'
        """
        if method not in {"counting", "sc", "sii"}:
            raise ValueError("method must be one of {'counting','sc','sii'}")
        base = os.path.join(DATA_DIR, "tensors", dataset)

        vocab_path = os.path.join(base, f"vocabularies/{dims}.pkl")
        populated_path = os.path.join(base,"populated", f"{method}_{dims}.pt")
        if not os.path.exists(vocab_path):
            raise FileNotFoundError(f"Missing vocab file: {vocab_path}")
        if not os.path.exists(populated_path):
            raise FileNotFoundError(f"Missing decomposition file: {populated_path}")
        # the vocab is here under f"vocabularies_[dims].pkl"
        # Load with torch (they were saved with torch.save)
        with open(vocab_path, "rb") as f:
            vocab = pickle.load(f)
        tensor = _torch_or_tucker_load(populated_path, map_location=map_location)

        return cls(tensor, device=map_location, sparsity_type=sparsity_type)



    # -- Sparsity methods ---
    def sparse_representation(self, sparse_type):
        # we return the sparse representation of the tensor
        if sparse_type == self.sparsity_type:
            return self.tensor
        # we check if our tensor is a tensorflow tensor or make it one
        if sparse_type == "tensorflow":
            if self.sparsity_type != "torch":
                tensor = self.sparse_representation("torch")
            else:
                tensor = self.tensor
            # we build from torch sparse tensor
            indices = tensor.coalesce().indices().t().numpy()   # shape (nnz, ndim)
            values  = tensor.coalesce().values().numpy()        # shape (nnz,)
            shape   = tuple(self.shape)          # e.g. (d0, d1, ..., d_{n-1})
            sparse_tensor = tf.SparseTensor(indices=indices, values=values, dense_shape=shape)
            # we warn users that tensorflow sparse tensors map directly to GPU.
            # additionally, they directly "allocate" the whole GPU memory to tf to reduce fragmentation later on.
            # this makes nvtop commands etc. not useable anymore

            print("WARNING: TensorFlow sparse tensors are allocated on GPU and may reserve large amounts of GPU memory.")

            return sparse_tensor

        elif sparse_type == "torch":
            if not self.sparsity_type or self.sparsity_type == "dense":
               return self.tensor.to_sparse()
            # can work from any tensor-like object
            elif self.sparsity_type == "cupy":
                return cupy_to_torch_sparse(self.tensor, orig_shape=self.shape)
            elif self.sparsity_type == "tensorflow":
                coords = self.tensor.indices.numpy()       # shape (nnz, ndim)
                data   = self.tensor.values.numpy()        # shape (nnz,)
                shape  = tuple(self.shape)  # e.g. (d0, d1, ..., d_{n-1})
                sparse_tensor = torch.sparse_coo_tensor(torch.tensor(coords).t(), torch.tensor(data), size=shape, device="cpu")
                return sparse_tensor
            else:
                raise NotImplementedError("sparsity_type must be one of {'dense', None, 'cupy', 'tensorflow','torch'}")

        elif sparse_type == "sparse":
            # can only work from a sparse torch tensor
            if not isinstance(self.tensor, torch.Tensor) or not self.tensor.is_sparse:
                raise TypeError("sparse expects self.tensor to be a torch sparse tensor.")
            coords = self.tensor.indices().numpy()       # shape (nnz, ndim)
            data   = self.tensor.values().numpy()        # shape (nnz,)
            shape  = tuple(self.tensor.size())  # e.g. (d0, d1, ..., d_{n-1})
            sparse_tensor = sparse.COO(coords, data, shape=shape)
            return sparse_tensor

        elif sparse_type == "cupy":
            if not isinstance(self.tensor, torch.Tensor) or not self.tensor.is_sparse:
                raise TypeError("cupy expects self.tensor to be a torch sparse tensor.")
            tensor_cupy, shape = torch_sparse_to_cupy(self.tensor)
            return tensor_cupy
        else:
            raise NotImplementedError(f"Sparse representation for type {sparse_type} not implemented.")



    def tensor_to_sparse(self, sparse_type="tensorflow"):
        self.tensor = self.sparse_representation(sparse_type)
        self.sparsity_type = sparse_type
        if sparse_type in ["tensorflow", "cupy"]:
            self.device = "cuda"


    def tensor_to_dense(self):
        if not isinstance(self.tensor, torch.Tensor) or not self.tensor.is_sparse:
            raise TypeError("tensor_to_dense expects self.tensor to be a torch sparse tensor.")
        self.tensor = self.tensor.to_dense()
        self.sparsity_type = "dense"

    def to_device(self, device):
        self.tensor = tree_to_device(self.tensor, device)
        self.device = device
        if device == "cpu":
            torch.cuda.empty_cache()

    def inspect(self):
        print("type:", type(self.tensor))
        print("sparsity type:", self.sparsity_type)
        print("shape:", self.shape)
        print("device:", self.device)

        if not self.sparsity_type or self.sparsity_type == "dense":
            memory_size = self.tensor.element_size() * self.tensor.nelement()
        elif self.sparsity_type == "torch":
            nnz = self.tensor._nnz()
            dtype_size = self.tensor.values().element_size()
            memory_size = nnz * (self.tensor.indices().element_size() * self.tensor.indices().shape[0] + dtype_size)

        elif self.sparsity_type == "cupy":
            memory_size = self.tensor.data.nbytes + self.tensor.row.nbytes + self.tensor.col.nbytes
        elif self.sparsity_type == "sparse":
            memory_size = self.tensor.nbytes
        elif self.sparsity_type == "tensorflow":
            nnz = self.tensor.values.shape[0]
            dtype_size = self.tensor.values.dtype.size
            memory_size = nnz * (self.tensor.indices.dtype.size * self.tensor.indices.shape[1] + dtype_size)

        else:
            memory_size = self.tensor.nbytes
        print(f"approx. memory size: {memory_size / (1024**2):.2f} MB")



In [41]:
dense_tensor = SparseTupleTensor.load_from_disk(dataset="fineweb_sparse",
                                          method="sii",
                                          dims=1000,
                                          map_location="cpu",
                                          sparsity_type="torch")
print("Initial tensor:")
dense_tensor.inspect()
print("\nConverting to dense representation:")
dense_tensor.tensor_to_dense()
dense_tensor.to_device("cuda")
dense_tensor.inspect()


Initial tensor:
type: <class 'torch.Tensor'>
sparsity type: torch
shape: torch.Size([1000, 1000, 1000])
device: cpu
approx. memory size: 5.94 MB

Converting to dense representation:
type: <class 'torch.Tensor'>
sparsity type: dense
shape: torch.Size([1000, 1000, 1000])
device: cuda
approx. memory size: 3814.70 MB


## 0. Initializing rank and epsilon (can be taken over from cupy implementation)


In [42]:
import tensorly_custom as tl
from tensorly_custom.tucker_tensor import validate_tucker_rank
tl.set_backend("pytorch")
# we import "validate tucker rank"
shape = tuple(dense_tensor.shape)
tensor = dense_tensor.tensor
rank = validate_tucker_rank(shape, rank=(50, 50, 50))
print("Using rank:", rank)
modes = list(range(len(rank)))
print("Using modes:", modes)
non_negative = True


Using rank: (50, 50, 50)
Using modes: [0, 1, 2]


## 1. Intializing core and factors

In [43]:
init = "random"
random_state = 42


if init == "random":
    rng = tl.check_random_state(random_state)
    core = tl.tensor(
        rng.random_sample([rank[index] for index in range(len(modes))]) + 0.01,
        **tl.context(tensor),
    )  # Check this
    factors = [
        tl.tensor(
            rng.random_sample((tensor.shape[mode], rank[index])),
            **tl.context(tensor),
        )
        for index, mode in enumerate(modes)
    ]

else:
    (core, factors) = init

if non_negative is True:
    factors = [tl.abs(f) for f in factors]
    core = tl.abs(core)

In [44]:
norm_tensor = tl.norm(tensor, 2)

In [45]:
norm_tensor

tensor(923.1670, device='cuda:0')

## 2. Iteration
### 2.1 mode iteration
We do the steps for a single mode

In [46]:
from tensorly_custom.decomposition._tucker import tucker_to_tensor
from tensorly_custom.base import unfold
# the first steps can be done with the dense
epsilon = 1e-12
for mode in modes:
    print("Updating mode:", mode)
    B = tucker_to_tensor((core, factors), skip_factor=mode)
    B = tl.transpose(unfold(B, mode))
    B.shape
    unfolded = unfold(tensor, mode)
    numerator = tl.dot(unfolded, B)
    numerator = tl.clip(numerator, a_min=epsilon, a_max=None)
    denominator = tl.dot(factors[mode], tl.dot(tl.transpose(B), B))
    denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
    factors[mode] *= numerator / denominator


Updating mode: 0
tucker_to_tensor time: 0.0012316703796386719
Updating mode: 1
tucker_to_tensor time: 9.083747863769531e-05
Updating mode: 2
tucker_to_tensor time: 9.393692016601562e-05


## 2.2 core update
Here comes the hard part (in sparse cupy)
tucker_to_tensor implies recombining the original tensor and the new factors

In [47]:
numerator = tucker_to_tensor((tensor, factors), transpose_factors=True)
numerator.shape

tucker_to_tensor time: 0.00985097885131836


torch.Size([50, 50, 50])

In [49]:
numerator = tl.clip(numerator, a_min=epsilon, a_max=None)

In [50]:
from tensorly_custom.tenalg import mode_dot
for i, f in enumerate(factors):
    if i:
        denominator = mode_dot(denominator, tl.dot(tl.transpose(f), f), i)
    else:
        denominator = mode_dot(core, tl.dot(tl.transpose(f), f), i)

In [51]:
denominator = tl.clip(denominator, a_min=epsilon, a_max=None)


In [52]:
core *= numerator / denominator

In [53]:
rec_error = (
            tl.norm(tensor - tucker_to_tensor((core, factors)), 2) / norm_tensor
        )

tucker_to_tensor time: 0.00975489616394043


In [55]:
rec_error

tensor(1.0000, device='cuda:0')